In [ ]:
device = "cuda:0"

HP_atomsdata = {"pml_rcut": 3.0, "pml_mnn": 12, "iml_rcut": 6.0, "iml_mnn": 24} # HP指hyperparams
HP_feat_dim = {'atom_dim': 64, 'bond_dim': 64, 'ang_dim': 32, 'dih_dim': 16}
HP_nn = {'init': 1, 'pml': 2, 'iml': 4, 'decoder': [64,1], 'pooling': 'sum'}
HP_train = {'batch_size': 8, 'max_epochs': 20, 'lr': 1e-2, 'adamw_weight_decay': 1e-3,
            'adamw_betas': (0.9, 0.999), '1cycle_final_div_factor': 1e+4, 'gradient_clip_val': 1.0}


In [ ]:
import os
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('../..')

from DIGNN.data import AtomsData, ase2AtomsData
from DIGNN.utils import AtomIndexMapper
from DIGNN.pl import DataModule, TrainModule_FF, TrainModule
from DIGNN.nn import models as dgm
from DIGNN.visualize import plot_comparison, plot_tsne, plot_umap


import time
import random
import torch
import numpy as np
import pytorch_lightning as pl
from ase.build import molecule

In [ ]:
from ase.io import read

qm7 = read('qm7_1000samples.xyz', index=':', format='extxyz')
for i in range(len(qm7)):
    qm7[i].arrays['energy'] = qm7[i].get_potential_energy()
    qm7[i].arrays['force'] = np.zeros_like(qm7[i].positions)
atomsdata = [ase2AtomsData(qm7[i], check_rcut=HP_atomsdata["pml_rcut"], properties=['energy','force']) for i in range(len(qm7))]


## 性质训练测试

In [ ]:
data = DataModule(atomsdata, 
                    **HP_atomsdata,
                    test_size=0.2, val_size=0.1,
                    batch_size=HP_train['batch_size'], num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='cplt',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

In [ ]:
# 模型定义
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **HP_feat_dim,
                                            pml_rcut=HP_atomsdata["pml_rcut"]+0.2,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            iml_rcut=HP_atomsdata["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**HP_feat_dim,
                                            pml=HP_nn['pml'],
                                            iml=HP_nn['iml'],
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            init_nn_layer=HP_nn['init'],
                                            ), 
                decoder=dgm.Decoder(dim=[HP_feat_dim['atom_dim']] + HP_nn['decoder'],
                                    reduce_method=HP_nn['pooling'],
                                    dropout=0.0),
                ).to(device)

In [ ]:
train_module = TrainModule(model, 
                           compile_model=False, # linux环境下开启编译，速度更快
                           lr=HP_train['lr'],
                           prop='energy',
                           adamw_weight_decay=HP_train['adamw_weight_decay'],
                           adamw_betas=HP_train['adamw_betas'],
                           onecycle_total_steps=HP_train['max_epochs']*len(data.train_dataloader()), 
                           onecycle_final_div_factor=HP_train['1cycle_final_div_factor'],
                           empty_cache_every_epoch=False,
                           enable_embed_decay=True,
                           )
trainer = pl.Trainer(max_epochs=HP_train['max_epochs'],
                    accelerator="gpu",
                    devices=[int(device.split(":")[-1])], # 单卡训练
                    check_val_every_n_epoch=1,
                    log_every_n_steps=100,
                    precision='16-mixed',
                    gradient_clip_val=HP_train['gradient_clip_val'],
                    benchmark=True,
                    # logger=tb_logger,
                    # callbacks=[checkpoint_callback],
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

In [ ]:
preds_ene, targets_ene = train_module.test_results.values()
plot_comparison(target=targets_ene, pred=preds_ene)

In [ ]:
features, labels = train_module.extract_features(
    dataloader=data.train_batch+data.val_batch+data.test_batch)

plot_tsne(features, labels, 
          perplexity=10 # 困惑度， 大样本时建议调大
          )


In [ ]:
plot_umap(features, labels, 
          n_neighbors=10,      # 控制局部/全局权衡。小值关注局部，大值关注全局
          min_dist=1          # 控制点之间的最小距离。小值更密，大值更稀疏
          )

## 力场训练测试

In [ ]:
data = DataModule(atomsdata, 
                    **hyperparams,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='basic',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

In [ ]:
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **DIGNN_feat_dim,
                                            pml_rcut=hyperparams["pml_rcut"]+0.2,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            iml_rcut=hyperparams["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**DIGNN_feat_dim,
                                            pml=1,
                                            iml=4,
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            init_nn_layer=0,
                                            ), 
                decoder=dgm.Decoder(dim=[DIGNN_feat_dim['atom_dim'],64,1], 
                                    reduce_method='sum', 
                                    dropout=0.0),
                ).to(device)

In [ ]:
max_epoch = 5

train_module = TrainModule_FF(model,
                              compile_model=False, # FF训练不开启
                              lr=1e-3,
                              energy_weight=0.1,
                              force_weight=1.0,
                              adamw_weight_decay=1e-2,
                              adamw_betas=(0.9, 0.999),
                              onecycle_total_steps=max_epoch*len(data.train_dataloader()),
                              onecycle_final_div_factor=1e+5,
                              )
trainer = pl.Trainer(max_epochs=max_epoch,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=50,
                    benchmark=True,
                    inference_mode=False,
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

preds_ene, targets_ene, preds_force, targets_force, atom_num = train_module.test_results.values()
plot_comparison(target=targets_ene, pred=preds_ene, atom_num=atom_num)
# plot_comparison(target=targets_force, pred=preds_force)